In [1]:
import os
os.chdir('/Users/jeffreybloodworth/Desktop/git-repos/scRNAseq-ExplainableML')
os.getcwd()

'/Users/jeffreybloodworth/Desktop/git-repos/scRNAseq-ExplainableML'

In [2]:
from pathlib import Path
import pandas as pd

all_pdcs = []

for meta_file in Path("data/raw").glob("*metadata*.gz"):

    meta = pd.read_csv(meta_file, sep="\t")

    pdc = meta[
        meta["CellType"] == "pDC"
    ].copy()

    pdc["sample"] = (
        meta_file.name
        .replace("_metadata.tsv.gz", "")
    )

    all_pdcs.append(pdc)

pdcs = pd.concat(
    all_pdcs,
    ignore_index=True
)

print(pdcs.shape)

(14430, 58)


In [3]:
pdcs["State"] = pdcs["is_malignant"]

In [4]:
pdcs["State"].value_counts()

State
Malignant       13732
Premalignant      495
Healthy           203
Name: count, dtype: int64

In [5]:
pdcs["Patient"] = (
    pdcs["sample"]
    .str.extract(r"(Pt\d+)")
)

In [6]:
pdcs["Patient"] = (
    pdcs["Patient"]
    .fillna("Healthy")
)

In [7]:
pdcs.groupby(
    ["sample", "State"]
).size()

sample             State       
GSE227690_BM       Healthy          203
GSE227690_Pt10Dx   Malignant         19
                   Premalignant      40
GSE227690_Pt10Rel  Malignant       4382
GSE227690_Pt12Dx   Malignant          2
                   Premalignant     265
GSE227690_Pt12Rel  Malignant       1041
GSE227690_Pt14Dx   Malignant       1095
GSE227690_Pt15Dx   Malignant       3029
GSE227690_Pt16Dx   Malignant       4031
GSE227690_Pt1Dx    Malignant        131
GSE227690_Pt1Rem   Premalignant      15
GSE227690_Pt5Dx    Premalignant      39
GSE227690_Pt9Dx    Malignant          2
                   Premalignant     136
dtype: int64

In [8]:
sample_summary = (
    pdcs.groupby(
        ["sample", "State"]
    )
    .size()
    .reset_index(name="cells")
)

sample_summary


,sample,State,cells
0,GSE227690_BM,Healthy,203
1,GSE227690_Pt10Dx,Malignant,19
2,GSE227690_Pt10Dx,Premalignant,40
3,GSE227690_Pt10Rel,Malignant,4382
4,GSE227690_Pt12Dx,Malignant,2
5,GSE227690_Pt12Dx,Premalignant,265
6,GSE227690_Pt12Rel,Malignant,1041
7,GSE227690_Pt14Dx,Malignant,1095
8,GSE227690_Pt15Dx,Malignant,3029
9,GSE227690_Pt16Dx,Malignant,4031


In [9]:
meta = pd.read_csv(
    "data/raw/GSE227690_Pt10Dx_metadata.tsv.gz",
    sep="\t"
)

counts = pd.read_csv(
    "data/raw/GSE227690_Pt10Dx_counts.tsv.gz",
    sep="\t",
    index_col=0
)


In [10]:
pdc_cells = meta.loc[
    meta["CellType"] == "pDC",
    "cell"
]

In [11]:
pdc_counts = counts[pdc_cells]

In [12]:
pdc_counts.shape


(16955, 59)

In [13]:
from pathlib import Path

import pandas as pd
import numpy as np

In [14]:
all_pdc_meta = []

for meta_file in Path("data/raw").glob("*metadata*.gz"):

    meta = pd.read_csv(
        meta_file,
        sep="\t"
    )

    pdc = meta[
        meta["CellType"] == "pDC"
    ].copy()

    pdc["sample"] = (
        meta_file.name
        .replace("_metadata.tsv.gz", "")
    )

    all_pdc_meta.append(pdc)

pdc_meta = pd.concat(
    all_pdc_meta,
    ignore_index=True
)

print(pdc_meta.shape)

(14430, 58)


In [15]:
pdc_meta["is_malignant"].value_counts()

is_malignant
Malignant       13732
Premalignant      495
Healthy           203
Name: count, dtype: int64

In [16]:
pdc_meta.to_csv(
    "data/processed/pdc_metadata.csv",
    index=False
)

In [17]:
import os

os.path.getsize(
    "data/raw/GSE227690_Pt16Dx_counts.tsv.gz"
)

4248122

In [18]:
from pathlib import Path

count_files = list(
    Path("data/raw").glob("*_counts.tsv.gz")
)

len(count_files)

7

In [19]:
for f in sorted(count_files):
    print(f.name)

GSE227690_BM_counts.tsv.gz
GSE227690_Pt10Dx_counts.tsv.gz
GSE227690_Pt10Rel_counts.tsv.gz
GSE227690_Pt12Dx_counts.tsv.gz
GSE227690_Pt15Dx_counts.tsv.gz
GSE227690_Pt16Dx_counts.tsv.gz
GSE227690_Pt9Dx_counts.tsv.gz


In [20]:
from pathlib import Path
import pandas as pd

all_pdc_meta = []

for meta_file in Path("data/raw").glob("*metadata*.gz"):

    meta = pd.read_csv(
        meta_file,
        sep="\t"
    )

    pdc = meta[
        meta["CellType"] == "pDC"
    ].copy()

    pdc["sample"] = (
        meta_file.name
        .replace("_metadata.tsv.gz", "")
    )

    all_pdc_meta.append(pdc)

pdc_meta = pd.concat(
    all_pdc_meta,
    ignore_index=True
)

print(pdc_meta.shape)

pdc_meta["is_malignant"].value_counts()

(14430, 58)


is_malignant
Malignant       13732
Premalignant      495
Healthy           203
Name: count, dtype: int64

In [21]:
pdc_meta.to_csv(
    "data/processed/pdc_metadata.csv",
    index=False
)

In [22]:
from pathlib import Path

all_counts = []

for meta_file in Path("data/raw").glob("*metadata*.gz"):

    count_file = Path(
        str(meta_file).replace(
            "_metadata.tsv.gz",
            "_counts.tsv.gz"
        )
    )

    if not count_file.exists():

        print(
            f"Skipping missing file: {count_file.name}"
        )

        continue

    print(
        f"\nProcessing {count_file.name}"
    )

    try:

        meta = pd.read_csv(
            meta_file,
            sep="\t"
        )

        counts = pd.read_csv(
            count_file,
            sep="\t",
            index_col=0
        )

    except Exception as e:

        print(
            f"FAILED: {count_file.name}"
        )

        print(e)

        continue

    pdc_cells = (
        meta.loc[
            meta["CellType"] == "pDC",
            "cell"
        ]
        .tolist()
    )

    pdc_cells = [
        cell
        for cell in pdc_cells
        if cell in counts.columns
    ]

    pdc_counts = counts[pdc_cells]

    print(
        f"Keeping {len(pdc_cells)} pDC cells"
    )

    all_counts.append(
        pdc_counts
    )

Skipping missing file: GSE227690_Pt12Rel_counts.tsv.gz
Skipping missing file: GSE227690_Pt5Dx_counts.tsv.gz
Skipping missing file: GSE227690_Pt1Dx_counts.tsv.gz
Skipping missing file: GSE227690_Pt1Rem_counts.tsv.gz

Processing GSE227690_Pt16Dx_counts.tsv.gz
FAILED: GSE227690_Pt16Dx_counts.tsv.gz
Error tokenizing data. C error: Expected 4453 fields in line 1866, saw 5364


Processing GSE227690_Pt9Dx_counts.tsv.gz
Keeping 138 pDC cells

Processing GSE227690_Pt12Dx_counts.tsv.gz
Keeping 267 pDC cells
Skipping missing file: GSE227690_Pt14Dx_counts.tsv.gz

Processing GSE227690_BM_counts.tsv.gz
FAILED: GSE227690_BM_counts.tsv.gz
Error tokenizing data. C error: Expected 20412 fields in line 2451, saw 23680


Processing GSE227690_Pt15Dx_counts.tsv.gz
FAILED: GSE227690_Pt15Dx_counts.tsv.gz
Error tokenizing data. C error: Expected 3471 fields in line 2400, saw 5124


Processing GSE227690_Pt10Rel_counts.tsv.gz
Keeping 4382 pDC cells

Processing GSE227690_Pt10Dx_counts.tsv.gz
Keeping 59 pDC cells


In [23]:
combined_counts = pd.concat(
    all_counts,
    axis=1
)

In [24]:
combined_counts.shape

(16955, 4846)

In [25]:
pdc_meta = pdc_meta.rename(
    columns={
        "index": "cell"
    }
)

In [26]:
print(
    pdc_meta.columns.tolist()
)

['cell', 'orig.ident', 'orig.ident2', 'nCount_RNA', 'nFeature_RNA', 'replicate', 'tech', 'CellType', 'project.umap.x', 'project.umap.y', 'MALAT1.chr11:65270399:G/A', 'NFIC.chr19:3462800:G/A', 'NOTCH1.chr9:139389423:C/T', 'TET2.Q1537*', 'bm_involvement', 'is_malignant', 'sample', 'TET2.N281fs*', 'TET2.V1395fs*', 'ZRSR2.Y175*', 'ASXL1.L697fs*', 'FAM98C.chr19:38899400:G/A', 'SETX.chr9:135136947:G/A', 'SMARCC1.chr3:47627735:G/A', 'TET2.Q960*', 'ASXL1.R693*', 'TET2.Q876*', 'TET2.S794*', 'ZRSR2.R126*', 'CUX1.L911fs*', 'TET2.E1437fs*', 'TET2.Q1547*', 'ETV6.R369W', 'EZH2.P132L', 'IDH2.R140Q', 'UMAP_1', 'UMAP_2', 'S.Score', 'G2M.Score', 'Phase', 'ASXL1.V807fs*', 'TET2.R1516*', 'U2AF1.S34F', 'ACAP2.L97M', 'ASXL1.G642fs', 'CWF19L2.K221R', 'DOLPP1.R227S', 'HNRNPUL1.F559F', 'MAP4K5.P667S', 'RAB9A.3pUTR', 'TET2.H1380Y', 'TET2.Q1034*', 'TET2.R1216*', 'TET2.S792*', 'MTAP.rearr', 'PRKDC.chr8:48713410:C/T', 'RPS24.chr10:79795273:T/C', 'SMARCD1.chr12:50490729:C/T']


In [27]:
pdc_meta = pdc_meta[
    pdc_meta["cell"].isin(
        combined_counts.columns
    )
].copy()

In [28]:
pdc_meta = (
    pdc_meta
    .set_index("cell")
    .loc[
        combined_counts.columns
    ]
    .reset_index()
)

In [29]:
all(
    pdc_meta["cell"]
    ==
    combined_counts.columns
)


KeyError: 'cell'

In [ ]:
print(pdc_meta.columns.tolist())
print(pdc_meta.head())

In [ ]:
print(pdc_meta.columns[0])

print(pdc_meta.iloc[:5,0])

print(combined_counts.columns[:5])

In [ ]:
barcode_col = pdc_meta.columns[0]

all(
    pdc_meta[barcode_col].values
    ==
    combined_counts.columns.values
)

In [ ]:
print(pdc_meta.index.name)

In [ ]:
print(pdc_meta.columns.tolist())
print(pdc_meta.index.name)

In [ ]:
all(
    pdc_meta["index"].values
    ==
    combined_counts.columns.values
)

In [ ]:
pdc_meta = pdc_meta.rename(
    columns={
        "index": "cell"
    }
)

In [ ]:
pdc_meta.columns[0]

In [ ]:
combined_counts.to_csv(
    "data/processed/pdc_counts.csv.gz",
    compression="gzip"
)

pdc_meta.to_csv(
    "data/processed/pdc_metadata.csv",
    index=False
)